In [ ]:
import json
from pathlib import Path

import numpy as np


DATA_DIR = Path.home() / "genome"


# ============================================================
# Utilities
# ============================================================

def load_json(path):
    print(f"Loading {path.name}...")

    with open(path, "r") as f:
        return json.load(f)


def zscore(values):
    values = np.asarray(values, dtype=np.float64)

    std = values.std()

    if std == 0:
        return np.zeros_like(values)

    return (values - values.mean()) / std


# ============================================================
# Build samples
# ============================================================

def build_samples(data_dir):

    image_data = load_json(
        data_dir / "image_data.json"
    )

    region_descriptions = load_json(
        data_dir / "region_descriptions.json"
    )

    objects = load_json(
        data_dir / "objects.json"
    )

    attributes = load_json(
        data_dir / "attributes.json"
    )

    relationships = load_json(
        data_dir / "relationships.json"
    )

    # --------------------------------------------------------
    # COCO IDs
    # --------------------------------------------------------

    coco_ids = {
        item["image_id"]: item.get("coco_id")
        for item in image_data
    }

    # --------------------------------------------------------
    # Objects
    # --------------------------------------------------------

    objects_by_id = {
        item["image_id"]: item.get("objects", [])
        for item in objects
    }

    # --------------------------------------------------------
    # Attributes
    # --------------------------------------------------------

    attributes_by_id = {
        item["image_id"]: item.get("attributes", [])
        for item in attributes
    }

    # --------------------------------------------------------
    # Relationships
    # --------------------------------------------------------

    relationships_by_id = {
        item["image_id"]: item.get("relationships", [])
        for item in relationships
    }

    # --------------------------------------------------------
    # Region descriptions
    #
    # IMPORTANT:
    # region_descriptions.json has:
    #
    # {
    #     "id": 1,
    #     "regions": [...]
    # }
    #
    # not:
    #
    # {
    #     "image_id": 1,
    #     ...
    # }
    # --------------------------------------------------------

    regions_by_id = {
        item["id"]: item.get("regions", [])
        for item in region_descriptions
    }

    # --------------------------------------------------------
    # Build samples
    # --------------------------------------------------------

    samples = []

    for image_id, regions in regions_by_id.items():

        # ----------------------------------------------------
        # Remove images that overlap with COCO
        # ----------------------------------------------------

        coco_id = coco_ids.get(image_id)

        if coco_id is not None:
            print("coco")
            continue

        # ----------------------------------------------------
        # Number of objects
        # ----------------------------------------------------

        n_objects = len(
            objects_by_id.get(image_id, [])
        )

        # ----------------------------------------------------
        # Number of attributes
        # ----------------------------------------------------

        n_attributes = 0

        for obj in attributes_by_id.get(
            image_id, []
        ):
            n_attributes += len(
                obj.get("attributes", [])
            )

        # ----------------------------------------------------
        # Number of relationships
        # ----------------------------------------------------

        n_relationships = len(
            relationships_by_id.get(
                image_id, []
            )
        )

        # ----------------------------------------------------
        # Number of region descriptions
        # ----------------------------------------------------

        n_regions = len(regions)

        samples.append(
            {
                "image_id": image_id,
                "coco_id": None,
                "n_objects": n_objects,
                "n_attributes": n_attributes,
                "n_relationships": n_relationships,
                "n_regions": n_regions,
            }
        )

    return samples


# ============================================================
# Complexity-based split
# ============================================================

def create_complexity_split(
    samples,
    n_id=15000,
    n_ood=1500,
):
    if len(samples) < n_id + n_ood:
        raise ValueError(
            f"Not enough samples: found {len(samples):,}, "
            f"but need {n_id + n_ood:,}."
        )

    # --------------------------------------------------------
    # Extract complexity components
    # --------------------------------------------------------

    objects = [
        x["n_objects"]
        for x in samples
    ]

    attributes = [
        x["n_attributes"]
        for x in samples
    ]

    relationships = [
        x["n_relationships"]
        for x in samples
    ]

    regions = [
        x["n_regions"]
        for x in samples
    ]

    # --------------------------------------------------------
    # Standardize each component
    # --------------------------------------------------------

    z_objects = zscore(objects)
    z_attributes = zscore(attributes)
    z_relationships = zscore(relationships)
    z_regions = zscore(regions)

    # --------------------------------------------------------
    # Semantic complexity score
    # --------------------------------------------------------

    # complexity = (
    #     z_objects
    #     + z_attributes
    #     + z_relationships
    #     + z_regions
    # ) / 4.0
    
    complexity = z_regions

    # --------------------------------------------------------
    # Attach complexity
    # --------------------------------------------------------

    for sample, score in zip(
        samples,
        complexity,
    ):
        sample["complexity"] = float(score)

    # --------------------------------------------------------
    # Sort from lowest -> highest complexity
    # --------------------------------------------------------

    samples.sort(
        key=lambda x: x["complexity"]
    )

    # --------------------------------------------------------
    # Select exact numbers
    # --------------------------------------------------------

    id_samples = samples[:n_id]

    ood_samples = samples[-n_ood:]

    return id_samples, ood_samples


# ============================================================
# Save
# ============================================================

def save_json(data, path):

    with open(path, "w") as f:
        json.dump(
            data,
            f,
            indent=2,
        )

    print(f"Saved: {path}")


# ============================================================
# Statistics
# ============================================================

def print_statistics(
    name,
    samples,
):

    complexity = [
        x["complexity"]
        for x in samples
    ]

    print()
    print(name)
    print("-" * 50)

    print(
        f"Images:       {len(samples):,}"
    )

    print(
        f"Complexity:   {np.mean(complexity):.3f}"
    )

    print(
        f"Median:       {np.median(complexity):.3f}"
    )

    print(
        f"Objects:      "
        f"{np.mean([x['n_objects'] for x in samples]):.2f}"
    )

    print(
        f"Attributes:   "
        f"{np.mean([x['n_attributes'] for x in samples]):.2f}"
    )

    print(
        f"Relations:    "
        f"{np.mean([x['n_relationships'] for x in samples]):.2f}"
    )

    print(
        f"Regions:      "
        f"{np.mean([x['n_regions'] for x in samples]):.2f}"
    )


# ============================================================
# Main
# ============================================================

def main():

    samples = build_samples(
        DATA_DIR
    )

    print()
    print(
        f"Remaining after COCO removal: "
        f"{len(samples):,}"
    )

    id_samples, ood_samples = (
        create_complexity_split(
            samples,
        )
    )

    print_statistics(
        "ID",
        id_samples,
    )

    print_statistics(
        "OOD",
        ood_samples,
    )

    save_json(
        id_samples,
        DATA_DIR / "id.json",
    )

    save_json(
        ood_samples,
        DATA_DIR / "ood.json",
    )


if __name__ == "__main__":
    main()

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image


DATA_DIR = Path.home() / "genome"

N_SAMPLES = 8
SEED = 30


def load_ids(filename):
    with open(DATA_DIR / filename, "r") as f:
        data = json.load(f)

    return [item["image_id"] for item in data]


def find_image(image_id):
    """
    Visual Genome stores images across VG_100K and VG_100K_2.
    """

    candidates = [
        DATA_DIR / "VG_100K" / f"{image_id}.jpg",
        DATA_DIR / "VG_100K_2" / f"{image_id}.jpg",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


def plot_samples(ids, title, n_samples=8, seed=42):

    rng = random.Random(seed)

    # Only keep images that actually exist
    available = [
        image_id
        for image_id in ids
        if find_image(image_id) is not None
    ]

    print(
        f"{title}: {len(available):,} images available "
        f"out of {len(ids):,}"
    )

    if len(available) < n_samples:
        raise ValueError(
            f"Only {len(available)} images are available."
        )

    selected = rng.sample(
        available,
        n_samples,
    )

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(16, 8),
    )

    axes = axes.flatten()

    for ax, image_id in zip(axes, selected):

        image_path = find_image(image_id)

        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)
        ax.set_title(f"ID: {image_id}")
        ax.axis("off")

    fig.suptitle(
        title,
        fontsize=16,
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# Load splits
# ============================================================

id_ids = load_ids("id.json")
ood_ids = load_ids("ood.json")

print(f"ID samples:  {len(id_ids):,}")
print(f"OOD samples: {len(ood_ids):,}")


# ============================================================
# Plot
# ============================================================

plot_samples(
    id_ids,
    "Visual Genome — ID Samples",
    n_samples=N_SAMPLES,
    seed=SEED,
)

plot_samples(
    ood_ids,
    "Visual Genome — OOD Samples",
    n_samples=N_SAMPLES,
    seed=SEED,
)

In [ ]:
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

DATA_DIR = Path.home() / "genome"

BATCH_SIZE = 64
NUM_WORKERS = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Dataset
# ============================================================

class VisualGenomeDataset(Dataset):

    def __init__(self, image_ids, data_dir, transform):
        self.image_ids = image_ids
        self.data_dir = data_dir
        self.transform = transform

    def find_image(self, image_id):

        candidates = [
            self.data_dir / "VG_100K" / f"{image_id}.jpg",
            self.data_dir / "VG_100K_2" / f"{image_id}.jpg",
        ]

        for path in candidates:
            if path.exists():
                return path

        raise FileNotFoundError(
            f"Image not found: {image_id}"
        )

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):

        image_id = self.image_ids[idx]

        image_path = self.find_image(image_id)

        image = Image.open(
            image_path
        ).convert("RGB")

        image = self.transform(image)

        return image, image_id


# ============================================================
# Load split
# ============================================================

def load_ids(filename):

    with open(DATA_DIR / filename, "r") as f:
        data = json.load(f)

    return [
        item["image_id"]
        for item in data
    ]


# ============================================================
# Extract representations
# ============================================================

@torch.no_grad()
def extract_representations(
    model,
    dataloader,
    device,
):

    representations = []
    image_ids = []

    for images, ids in tqdm(
        dataloader,
        desc="Extracting",
    ):

        images = images.to(
            device,
            non_blocking=True,
        )

        # ResNet-50 output after global average pooling:
        # [B, 2048, 1, 1]
        features = model(images)

        # [B, 2048]
        features = features.flatten(1)

        representations.append(
            features.cpu()
        )

        image_ids.extend(
            ids.tolist()
        )

    representations = torch.cat(
        representations,
        dim=0,
    )

    return representations, image_ids


# ============================================================
# Main
# ============================================================

def main():

    print(f"Device: {DEVICE}")

    # --------------------------------------------------------
    # ImageNet preprocessing
    # --------------------------------------------------------

    weights = ResNet50_Weights.IMAGENET1K_V2
    transform = weights.transforms()

    # --------------------------------------------------------
    # ResNet-50
    #
    # Remove only the final classifier.
    #
    # Output:
    # [B, 2048, 1, 1]
    # --------------------------------------------------------

    resnet = resnet50(
        weights=weights
    )

    resnet = torch.nn.Sequential(
        *list(resnet.children())[:-1]
    )

    resnet.eval()
    resnet.to(DEVICE)

    # --------------------------------------------------------
    # Load splits
    # --------------------------------------------------------

    id_ids = load_ids("id.json")
    ood_ids = load_ids("ood.json")

    print(
        f"ID images:  {len(id_ids):,}"
    )

    print(
        f"OOD images: {len(ood_ids):,}"
    )

    # --------------------------------------------------------
    # Extract representations
    # --------------------------------------------------------

    for split_name, image_ids in [
        ("id", id_ids),
        ("ood", ood_ids),
    ]:

        dataset = VisualGenomeDataset(
            image_ids=image_ids,
            data_dir=DATA_DIR,
            transform=transform,
        )

        dataloader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )

        print(
            f"\nExtracting {split_name}..."
        )

        representations, image_ids = (
            extract_representations(
                resnet,
                dataloader,
                DEVICE,
            )
        )

        print(
            f"Representation shape: "
            f"{representations.shape}"
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        output_path = (
            DATA_DIR
            / f"{split_name}_representations.pt"
        )

        torch.save(
            {
                "image_ids": image_ids,
                "representations": representations,
            },
            output_path,
        )

        print(
            f"Saved: {output_path}"
        )


if __name__ == "__main__":
    main()

In [51]:
data = torch.load(
    DATA_DIR / "ood_representations.pt",
    map_location="cpu",
)
data["representations"].dtype

torch.float32

In [81]:
import torch
import numpy as np
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
from pathlib import Path


# ============================================================
# 1. Load dataset
# ============================================================

DATA_DIR = Path.home() / "genome"

data = torch.load(
    DATA_DIR / "ood_representations.pt",
    map_location="cpu",
)

embeddings = data["representations"].numpy()
paths = data["image_ids"]

print("Total samples:", embeddings.shape)


# ============================================================
# 2. Normalize embeddings
# ============================================================

normalized_embeddings = embeddings / np.linalg.norm(
    embeddings,
    axis=1,
    keepdims=True,
)


# ============================================================
# 3. Configuration
# ============================================================

num_batches = 15
samples_per_batch = 100
n_samples = len(normalized_embeddings)

assert n_samples == num_batches * samples_per_batch, (
    f"Expected exactly {num_batches * samples_per_batch} "
    f"samples, got {n_samples}."
)


# ============================================================
# 4. Initial KMeans
# ============================================================

kmeans = KMeans(
    n_clusters=num_batches,
    random_state=42,
    n_init=10,
)

kmeans.fit(normalized_embeddings)

centroids = kmeans.cluster_centers_

# Normalize centroids
centroids /= np.linalg.norm(
    centroids,
    axis=1,
    keepdims=True,
)


# ============================================================
# 5. Balanced assignment
# ============================================================

# Cosine distance
distances = 1 - normalized_embeddings @ centroids.T

# We need to assign exactly 100 samples to each cluster.
#
# Expand each cluster 100 times.
#
# This creates:
#
#     1500 samples × 1500 slots
#
# Each cluster has exactly 100 slots.

slot_centroids = np.repeat(
    np.arange(num_batches),
    samples_per_batch,
)

cost_matrix = distances[
    :,
    slot_centroids,
]


# ============================================================
# 6. Globally optimal assignment
# ============================================================

row_ind, col_ind = linear_sum_assignment(
    cost_matrix
)

assert len(row_ind) == n_samples


# ============================================================
# 7. Convert assignments to batches
# ============================================================

cluster_ids = slot_centroids[col_ind]

batches = []
batch_paths = []

for batch_id in range(num_batches):

    idxs = np.where(
        cluster_ids == batch_id
    )[0]

    assert len(idxs) == samples_per_batch

    batch_embeddings = torch.tensor(
        embeddings[idxs],
        dtype=torch.float32,
    )

    batch_paths.append([
        paths[i]
        for i in idxs
    ])

    batches.append(
        batch_embeddings
    )

    print(
        f"Batch {batch_id}: "
        f"{len(idxs)} samples"
    )


# ============================================================
# 8. Verify NO duplicates
# ============================================================

all_indices = np.concatenate([
    np.where(cluster_ids == batch_id)[0]
    for batch_id in range(num_batches)
])

assert len(all_indices) == n_samples
assert len(np.unique(all_indices)) == n_samples

print("\nNo duplicated samples.")


# ============================================================
# 9. Verify ALL samples are used
# ============================================================

assert set(all_indices) == set(
    range(n_samples)
)

print("All samples are used.")


# ============================================================
# 10. Verify batch sizes
# ============================================================

assert all(
    len(batch) == samples_per_batch
    for batch in batches
)

print(
    f"All {num_batches} batches contain "
    f"exactly {samples_per_batch} samples."
)


# ============================================================
# 11. Save
# ============================================================

output_path = (
    DATA_DIR /
    "ood_similar_batches_representations.pt"
)

torch.save(
    {
        "representations": batches,
        "image_ids": batch_paths,
    },
    output_path,
)

print("\nSaved:", output_path)

Total samples: (1500, 2048)
Batch 0: 100 samples
Batch 1: 100 samples
Batch 2: 100 samples
Batch 3: 100 samples
Batch 4: 100 samples
Batch 5: 100 samples
Batch 6: 100 samples
Batch 7: 100 samples
Batch 8: 100 samples
Batch 9: 100 samples
Batch 10: 100 samples
Batch 11: 100 samples
Batch 12: 100 samples
Batch 13: 100 samples
Batch 14: 100 samples

No duplicated samples.
All samples are used.
All 15 batches contain exactly 100 samples.

Saved: /home/mehdi_jmlkh/genome/ood_similar_batches_representations.pt


In [78]:
def shuffle_samples_across_batches(batches, seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    batch_sizes = [batch.shape[0] for batch in batches]

    # Combine all samples
    all_samples = torch.cat(batches, dim=0)

    # Shuffle globally
    permutation = torch.randperm(
        len(all_samples),
        generator=generator,
    )

    all_samples = all_samples[permutation]

    # Split back into original batch sizes
    shuffled_batches = []

    start = 0

    for size in batch_sizes:
        shuffled_batches.append(
            all_samples[start:start + size]
        )
        start += size

    return shuffled_batches

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image


DATA_DIR = Path.home() / "genome"

N_IMAGES_PER_BATCH = 5
SEED = 42


# ============================================================
# Load saved batches
# ============================================================

data = torch.load(
    DATA_DIR / "ood_similar_batches_representations.pt",
    map_location="cpu",
)

batch_ids = data["image_ids"]

print(f"Number of batches: {len(batch_ids)}")


# ============================================================
# Find Visual Genome image
# ============================================================

def find_image(image_id):
    candidates = [
        DATA_DIR / "VG_100K" / f"{image_id}.jpg",
        DATA_DIR / "VG_100K_2" / f"{image_id}.jpg",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


# ============================================================
# Plot images from all batches
# ============================================================

rng = random.Random(SEED)

n_batches = len(batch_ids)

fig, axes = plt.subplots(
    n_batches,
    N_IMAGES_PER_BATCH,
    figsize=(16, 3 * n_batches),
)

axes = axes.reshape(n_batches, N_IMAGES_PER_BATCH)

for batch_id in range(n_batches):

    ids = batch_ids[batch_id]

    available = [
        image_id
        for image_id in ids
        if find_image(image_id) is not None
    ]

    print(
        f"Batch {batch_id}: "
        f"{len(available)}/{len(ids)} images available"
    )

    n = min(N_IMAGES_PER_BATCH, len(available))

    selected = rng.sample(available, n)

    for col in range(N_IMAGES_PER_BATCH):

        ax = axes[batch_id, col]
        ax.axis("off")

        if col >= n:
            continue

        image_id = selected[col]
        image_path = find_image(image_id)

        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            f"Batch {batch_id} | {image_id}",
            fontsize=9,
        )

    # Batch label on the left
    axes[batch_id, 0].set_ylabel(
        f"Batch {batch_id}",
        fontsize=12,
        rotation=90,
    )


fig.suptitle(
    "Samples from OOD Similarity Batches",
    fontsize=18,
)

plt.tight_layout()

plt.savefig(
    DATA_DIR / "ood_batch_image_samples.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
DATA_DIR = Path.home() / "genome"

id_data = torch.load(
    f"{DATA_DIR}/id_representations.pt",
    weights_only=False,
)

ood_data = torch.load(
    f"{DATA_DIR}/ood_representations.pt",
    weights_only=False,
)

print(id_data["representations"].shape)
print(ood_data["representations"].shape)

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path.home() / "genome"


def inspect_json(filename):
    path = DATA_DIR / filename

    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)

    with open(path, "r") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    if isinstance(data, list):
        print("Number of entries:", len(data))

        for i, item in enumerate(data[:3]):
            print(f"\n--- Entry {i} ---")
            print("Type:", type(item).__name__)

            if isinstance(item, dict):
                print("Keys:", list(item.keys()))
                print("Full entry:")
                print(item)
            else:
                print(item)

    elif isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))

        for key, value in list(data.items())[:3]:
            print(f"\n--- Key: {key} ---")
            print("Value type:", type(value).__name__)

            if isinstance(value, list):
                print("List length:", len(value))
                if value:
                    print("First item:")
                    print(value[0])
            else:
                print("Value:")
                print(value)


files = [
    "image_data.json",
    "region_descriptions.json",
    "objects.json",
    "attributes.json",
    "relationships.json",
]

for filename in files:
    inspect_json(filename)